# Análisis de errores, atajos visuales y Grad-CAM


## Celda 1 · Configuración de la sesión


In [1]:
from google.colab import drive, userdata
drive.mount('/content/drive')

token = userdata.get('GH_TOKEN')
![ -d /content/yuca_transfer ] || git clone https://{token}@github.com/Frank-Dev007/yuca_transfer.git /content/yuca_transfer
%cd /content/yuca_transfer
!git remote set-url origin https://{token}@github.com/Frank-Dev007/yuca_transfer.git
!git pull
!git config user.name "Frank Manuel Garcia"
!git config user.email "frankgpernett2006@gmail.com"

!python scripts/preparar_datos.py
!ls checkpoints results

Mounted at /content/drive
Cloning into '/content/yuca_transfer'...
remote: Enumerating objects: 329, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 329 (delta 32), reused 21 (delta 3), pack-reused 236 (from 1)
Receiving objects: 100% (329/329), 120.24 MiB | 32.72 MiB/s, done.
Resolving deltas: 100% (135/135), done.
/content/yuca_transfer
Already up to date.
Copiando desde Drive...
Tamaño del zip (MB): 1354.1
Contenido de data/: ['cassavaleafdata.zip', 'cassavaleafdata']
checkpoints:
E1_prueba.pt  E1.pt  E2.pt  E3.pt  E4.pt

results:
duplicados_exactos.csv	      E4
duplicados_phash_umbral0.csv  grupos_duplicados_exactos.csv
E0			      imagenes_excluidas.csv
E1			      inventario_imagenes.csv
E1_prueba		      particiones_finales.csv
E2			      tabla_consolidada.csv
E3


## Celda 2 · Preparar

In [2]:
import importlib, os, sys, json
os.chdir("/content/yuca_transfer"); sys.path.append("/content/yuca_transfer"); importlib.invalidate_caches()

import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import f1_score
from src.utils import fijar_semilla, obtener_dispositivo
from src.data import crear_dataloaders, transform_modelo, transform_base, CLASES
from src.models import crear_modelo

fijar_semilla(42)
dispositivo = obtener_dispositivo()
SALIDA = "results/analisis_errores"
os.makedirs(SALIDA, exist_ok=True)
loaders = crear_dataloaders(batch_size=32, seed=42)
test = loaders["test"].dataset
print(dispositivo, "| imágenes de test:", len(test))

cuda | imágenes de test: 1874


## Celda 3 · Análisis por clase en todos los experimentos


In [3]:
EXPS = ["E0", "E1", "E2", "E3", "E4"]
m = {e: json.load(open(f"results/{e}/metricas.json")) for e in EXPS}
recall = pd.DataFrame({e: {c: m[e]["test"]["por_clase"][c]["recall"] for c in CLASES} for e in EXPS}).round(3)
f1 = pd.DataFrame({e: {c: m[e]["test"]["por_clase"][c]["f1"] for c in CLASES} for e in EXPS}).round(3)
soporte = {c: m["E2"]["test"]["por_clase"][c]["soporte"] for c in CLASES}
recall.insert(0, "n test", pd.Series(soporte))
print("Recall por clase en test:"); display(recall)
print("F1 por clase en test:"); display(f1)
recall.to_csv(f"{SALIDA}/recall_por_clase_E0_E4.csv"); f1.to_csv(f"{SALIDA}/f1_por_clase_E0_E4.csv")

confusiones = {}
for e in EXPS:
    p = pd.read_csv(f"results/{e}/predicciones_test.csv")
    err = p[p["real"] != p.predicha]
    confusiones[e] = err.groupby(["real", "predicha"]).size().sort_values(ascending=False).head(5)
tabla_conf = pd.DataFrame({e: [f"{r} → {q}: {n}" for (r, q), n in confusiones[e].items()] for e in EXPS})
tabla_conf.index = [f"#{i+1}" for i in range(len(tabla_conf))]
print("Las 5 confusiones más frecuentes de cada modelo (real → predicha: cantidad):"); display(tabla_conf)
tabla_conf.to_csv(f"{SALIDA}/confusiones_frecuentes_E0_E4.csv")

Recall por clase en test:


,n test,E0,E1,E2,E3,E4
cbb,154,0.110,0.539,0.506,0.357,0.286
cbsd,477,0.419,0.795,0.778,0.713,0.748
cgm,253,0.032,0.723,0.739,0.735,0.648
cmd,885,0.901,0.832,0.903,0.892,0.906
healthy,105,0.067,0.571,0.686,0.743,0.667


F1 por clase en test:


,E0,E1,E2,E3,E4
cbb,0.173,0.476,0.523,0.425,0.365
cbsd,0.450,0.741,0.775,0.742,0.735
cgm,0.057,0.725,0.735,0.674,0.664
cmd,0.706,0.872,0.894,0.877,0.879
healthy,0.112,0.659,0.735,0.703,0.645


Las 5 confusiones más frecuentes de cada modelo (real → predicha: cantidad):


,E0,E1,E2,E3,E4
#1,cbsd → cmd: 245,cmd → cbsd: 71,cbsd → cbb: 48,cmd → cgm: 56,cbb → cbsd: 65
#2,cgm → cmd: 193,cbsd → cbb: 63,cgm → cmd: 42,cbsd → cmd: 51,cgm → cmd: 59
#3,cmd → cbsd: 76,cbb → cbsd: 50,cbb → cbsd: 42,cbb → cbsd: 48,cbsd → cmd: 50
#4,cbb → cmd: 74,cmd → cgm: 47,cmd → cbsd: 39,cgm → cmd: 44,cmd → cbsd: 35
#5,healthy → cmd: 65,cgm → cmd: 31,cmd → cgm: 33,cbsd → cbb: 40,cmd → cgm: 31


## Celda 4 · Predicciones de E2 con su confianza


In [4]:
modelo = crear_modelo("mobilenet_v2")
modelo.load_state_dict(torch.load("checkpoints/E2.pt", map_location=dispositivo))
modelo.to(dispositivo).eval()

probs = []
with torch.no_grad():
    for x, _ in loaders["test"]:
        probs.append(torch.softmax(modelo(x.to(dispositivo)), dim=1).cpu())
P = torch.cat(probs).numpy()
reales = np.array(test.etiquetas)

df = pd.DataFrame({"ruta": test.rutas, "archivo": [os.path.basename(r) for r in test.rutas],
                   "real": [CLASES[i] for i in reales], "predicha": [CLASES[i] for i in P.argmax(1)],
                   "confianza": P.max(1).round(3), "prob_clase_real": P[np.arange(len(P)), reales].round(3)})
for e in ["E1", "E3", "E4"]:                       # ¿los otros modelos aciertan esta misma imagen?
    p = pd.read_csv(f"results/{e}/predicciones_test.csv").set_index("ruta")
    df[f"acierta_{e}"] = [p.loc[r, "predicha"] == p.loc[r, "real"] for r in df.ruta]
df["acierta_E2"] = df["real"] == df.predicha
df.to_csv(f"{SALIDA}/predicciones_E2_con_confianza.csv", index=False)

f1_rec = f1_score(df["real"], df.predicha, average="macro")
print(f"macro-F1 recalculado: {f1_rec:.4f} | metricas.json de E2: {m['E2']['test']['macro_f1']:.4f}")
print(f"Errores de E2 en test: {(~df.acierta_E2).sum()} de {len(df)}")

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 77.4MB/s]


macro-F1 recalculado: 0.7323 | metricas.json de E2: 0.7323
Errores de E2 en test: 367 de 1874


## Celda 5 · Elegir los errores a analizar


In [5]:
N_PARES, POR_PAR = 4, 3
errores = df[~df.acierta_E2]
pares = errores.groupby(["real", "predicha"]).size().sort_values(ascending=False)
print("Confusiones más frecuentes de E2:"); display(pares.head(8).to_frame("cantidad"))

seleccion = pd.concat([errores[(errores["real"] == r) & (errores.predicha == q)].nlargest(POR_PAR, "confianza")
                       for (r, q) in pares.index[:N_PARES]])
seleccion.insert(0, "id", [f"ERR{i+1:02d}" for i in range(len(seleccion))])

persistentes = df[~(df.acierta_E1 | df.acierta_E2 | df.acierta_E3 | df.acierta_E4)]
print(f"\nErrores persistentes (fallan E1, E2, E3 y E4): {len(persistentes)} imágenes")
display(persistentes.groupby(["real", "predicha"]).size().sort_values(ascending=False).head(6).to_frame("cantidad"))
seleccion["persistente"] = seleccion.ruta.isin(persistentes.ruta)
display(seleccion[["id", "archivo", "real", "predicha", "confianza", "prob_clase_real", "acierta_E1", "acierta_E3", "acierta_E4", "persistente"]])

Confusiones más frecuentes de E2:


cantidad
real predicha          
cbsd cbb             48
cgm  cmd             42
cbb  cbsd            42
cmd  cbsd            39
     cgm             33
cbsd cmd             32
     cgm             22
cbb  cmd             20


Errores persistentes (fallan E1, E2, E3 y E4): 169 imágenes


,,cantidad
real,predicha,
cbb,cbsd,28
cbsd,cbb,19
cgm,cmd,18
cbsd,cmd,17
cbb,cmd,17
cmd,cbsd,11


,id,archivo,real,predicha,confianza,prob_clase_real,acierta_E1,acierta_E3,acierta_E4,persistente
282,ERR01,test-cbsd-215.jpg,cbsd,cbb,0.990,0.006,False,False,False,True
302,ERR02,test-cbsd-233.jpg,cbsd,cbb,0.954,0.044,False,True,True,False
389,ERR03,test-cbsd-311.jpg,cbsd,cbb,0.935,0.061,False,False,False,True
866,ERR04,test-cgm-83.jpg,cgm,cmd,0.990,0.009,False,True,False,False
855,ERR05,test-cgm-73.jpg,cgm,cmd,0.973,0.004,False,False,False,True
773,ERR06,test-cgm-23.jpg,cgm,cmd,0.966,0.029,False,False,False,True
21,ERR07,test-cbb-117.jpg,cbb,cbsd,0.976,0.023,False,False,False,True
151,ERR08,test-cbb-97.jpg,cbb,cbsd,0.962,0.017,False,False,False,True
125,ERR09,test-cbb-73.jpg,cbb,cbsd,0.959,0.036,False,False,False,True
1747,ERR10,test-cmd-877.jpg,cmd,cbsd,0.972,0.016,False,False,False,True


## Celda 6 · Ver cada error con Grad-CAM


In [6]:
class GradCAM:
    def __init__(self, modelo, capa):
        self.modelo, self.act, self.grad = modelo, None, None
        capa.register_forward_hook(self._guardar)
    def _guardar(self, modulo, entrada, salida):
        self.act = salida
        if salida.requires_grad:
            salida.register_hook(lambda g: setattr(self, "grad", g))
    def __call__(self, ruta, clase):
        img = Image.open(ruta).convert("RGB")
        x = transform_modelo(img).unsqueeze(0).to(dispositivo).requires_grad_(True)
        self.modelo.zero_grad()
        salida = self.modelo(x)
        salida[0, clase].backward()
        pesos = self.grad.mean(dim=(2, 3), keepdim=True)          # importancia de cada canal
        cam = torch.relu((pesos * self.act).sum(dim=1))[0]        # mapa 7x7
        cam = F.interpolate(cam[None, None], size=(224, 224), mode="bilinear")[0, 0]
        return (cam / (cam.max() + 1e-8)).detach().cpu().numpy()

gradcam = GradCAM(modelo, modelo.features[-1])

def fila_error(ejes, f):
    img = Image.open(f["ruta"]).convert("RGB")
    recorte = np.asarray(transform_base(img))
    ejes[0].imshow(img); ejes[0].set_title(f"{f['id']} · foto completa {img.size[0]}×{img.size[1]}", fontsize=9)
    ejes[1].imshow(recorte); ejes[1].set_title(f"real: {f['real']} · predicha: {f['predicha']} ({f['confianza']:.2f})", fontsize=9)
    for k, (clase, rotulo) in enumerate([(f["predicha"], "predicha"), (f["real"], "real")]):
        ejes[2 + k].imshow(recorte); ejes[2 + k].imshow(gradcam(f["ruta"], CLASES.index(clase)), cmap="jet", alpha=0.45)
        ejes[2 + k].set_title(f"Grad-CAM clase {rotulo}: {clase}", fontsize=9)
    for e in ejes: e.axis("off")

for parte, inicio in enumerate(range(0, len(seleccion), 4), start=1):
    bloque = seleccion.iloc[inicio:inicio + 4]
    fig, ejes = plt.subplots(len(bloque), 4, figsize=(16, 4.2 * len(bloque)), squeeze=False)
    for i, (_, f) in enumerate(bloque.iterrows()):
        fila_error(ejes[i], f)
    plt.tight_layout(); plt.savefig(f"{SALIDA}/errores_gradcam_parte{parte}.png", dpi=100); plt.show()

Output hidden; open in https://colab.research.google.com to view.

## Celda 7 · Escribir la causa de cada error (lo hace el equipo)


In [7]:
CAUSAS = {
    "ERR01": ("A", "El modelo toma la mancha marrón de la parte baja de la hoja como evidencia de cbb; la evidencia de cbsd aparece en el pasto del fondo, no en la hoja"),
    "ERR02": ("A", "Marca la hoja amarilla con zona seca como cbb; la evidencia de cbsd cae en la parte baja de esa misma hoja y en el fondo"),
    "ERR03": ("C", "La evidencia de cbsd está en la hoja izquierda con borde seco, que el recorte deja solo en parte; la de cbb se concentra en el centro, entre dos hojas"),
    "ERR04": ("A", "Las hojas moteadas del centro empujan hacia cmd; la evidencia de cgm aparece en la parte superior, que incluye suelo"),
    "ERR05": ("A", "El modelo se fija en la hoja enrollada de arriba (cmd); la evidencia de cgm está en una hoja del fondo y no en la hoja enfocada (confirmar)"),
    "ERR06": ("D", "La evidencia de cmd cae sobre sombras, tallos y suelo entre las hojas, con sol fuerte; posible atajo visual (confirmar)"),
    "ERR07": ("A", "Las manchas y los agujeros de la parte superior de la hoja empujan hacia cbsd; la evidencia de cbb está en los bordes de la hoja"),
    "ERR08": ("A", "Las manchas amarillas de la hoja superior empujan hacia cbsd; la evidencia de cbb está cerca del borde dañado del centro"),
    "ERR09": ("C", "La evidencia de cbsd se concentra en una sola hoja; la mancha marrón de la hoja izquierda queda fuera del recorte y las hojas en sombra no se tienen en cuenta (confirmar)"),
    "ERR10": ("D", "La evidencia de cbsd cae debajo de las hojas, sobre el suelo; la de cmd marca partes de las hojas pero también toca el suelo"),
    "ERR11": ("D", "La evidencia de cbsd incluye parte del fondo y ninguna zona aporta evidencia para cmd; foto muy alargada, el recorte elimina casi la mitad del ancho (confirmar si es G)"),
    "ERR12": ("E", "La evidencia de cmd es mínima y está sobre una zona en sombra; luz con mucho contraste; revisar si las manchas redondas corresponden a cmd (confirmar si es G o H)"),
}

seleccion["causa"] = seleccion["id"].map(lambda i: CAUSAS.get(i, ("", ""))[0])
seleccion["observacion"] = seleccion["id"].map(lambda i: CAUSAS.get(i, ("", ""))[1])
columnas = ["id", "archivo", "real", "predicha", "confianza", "prob_clase_real",
            "acierta_E1", "acierta_E3", "acierta_E4", "persistente", "causa", "observacion"]
seleccion[columnas].to_csv(f"{SALIDA}/errores_seleccionados.csv", index=False)
display(seleccion[columnas])
print("Errores con causa escrita:", (seleccion.causa != "").sum(), "de", len(seleccion))
if (seleccion.causa != "").sum():
    print(seleccion[seleccion.causa != ""].causa.value_counts())

,id,archivo,real,predicha,confianza,prob_clase_real,acierta_E1,acierta_E3,acierta_E4,persistente,causa,observacion
282,ERR01,test-cbsd-215.jpg,cbsd,cbb,0.990,0.006,False,False,False,True,A,El modelo toma la mancha marrón de la parte ba...
302,ERR02,test-cbsd-233.jpg,cbsd,cbb,0.954,0.044,False,True,True,False,A,Marca la hoja amarilla con zona seca como cbb;...
389,ERR03,test-cbsd-311.jpg,cbsd,cbb,0.935,0.061,False,False,False,True,C,La evidencia de cbsd está en la hoja izquierda...
866,ERR04,test-cgm-83.jpg,cgm,cmd,0.990,0.009,False,True,False,False,A,Las hojas moteadas del centro empujan hacia cm...
855,ERR05,test-cgm-73.jpg,cgm,cmd,0.973,0.004,False,False,False,True,A,El modelo se fija en la hoja enrollada de arri...
773,ERR06,test-cgm-23.jpg,cgm,cmd,0.966,0.029,False,False,False,True,D,"La evidencia de cmd cae sobre sombras, tallos ..."
21,ERR07,test-cbb-117.jpg,cbb,cbsd,0.976,0.023,False,False,False,True,A,Las manchas y los agujeros de la parte superio...
151,ERR08,test-cbb-97.jpg,cbb,cbsd,0.962,0.017,False,False,False,True,A,Las manchas amarillas de la hoja superior empu...
125,ERR09,test-cbb-73.jpg,cbb,cbsd,0.959,0.036,False,False,False,True,C,La evidencia de cbsd se concentra en una sola ...
1747,ERR10,test-cmd-877.jpg,cmd,cbsd,0.972,0.016,False,False,False,True,D,"La evidencia de cbsd cae debajo de las hojas, ..."


Errores con causa escrita: 12 de 12
causa
A    6
D    3
C    2
E    1
Name: count, dtype: int64


## Celda 8 · Grad-CAM en aciertos (control de atajos visuales)


In [8]:
aciertos = df[df.acierta_E2].sort_values("confianza", ascending=False).groupby("real").head(1)
fig, ejes = plt.subplots(2, len(aciertos), figsize=(4 * len(aciertos), 8.5), squeeze=False)
for j, (_, f) in enumerate(aciertos.iterrows()):
    recorte = np.asarray(transform_base(Image.open(f["ruta"]).convert("RGB")))
    ejes[0, j].imshow(recorte); ejes[0, j].set_title(f"{f['real']} · acierto ({f['confianza']:.2f})", fontsize=9)
    ejes[1, j].imshow(recorte); ejes[1, j].imshow(gradcam(f["ruta"], CLASES.index(f["real"])), cmap="jet", alpha=0.45)
    ejes[1, j].set_title("Grad-CAM", fontsize=9)
for e in ejes.flat: e.axis("off")
plt.tight_layout(); plt.savefig(f"{SALIDA}/aciertos_gradcam.png", dpi=100); plt.show()
aciertos[["archivo", "real", "confianza"]].to_csv(f"{SALIDA}/aciertos_gradcam.csv", index=False)

Output hidden; open in https://colab.research.google.com to view.

## Celda 9 · Dos pruebas rápidas de atajos y del recorte
- **Iluminación:** se divide test en 4 grupos según el brillo promedio de la foto y se mide la accuracy de E2 en cada grupo.
- **Recorte:** se agrupan las fotos según su proporción (cuadradas o alargadas). Las alargadas pierden más borde con el recorte central.

Si la accuracy cambia mucho entre grupos, es un indicio (no una prueba) de que la luz o el recorte influyen.

In [9]:
brillo, proporcion = [], []
for r in df.ruta:
    img = Image.open(r).convert("RGB")
    proporcion.append(max(img.size) / min(img.size))
    brillo.append(np.asarray(transform_base(img).convert("HSV"))[:, :, 2].mean() / 255)
df["brillo"], df["proporcion"] = np.round(brillo, 3), np.round(proporcion, 3)

df["grupo_brillo"] = pd.qcut(df.brillo, 4, labels=["1 más oscuras", "2", "3", "4 más claras"])
por_brillo = df.groupby("grupo_brillo", observed=True).agg(
    imagenes=("acierta_E2", "size"), accuracy_E2=("acierta_E2", "mean"), brillo_medio=("brillo", "mean")).round(3)
print("Accuracy de E2 según el brillo de la foto:"); display(por_brillo)

df["grupo_forma"] = pd.cut(df.proporcion, [0, 1.1, 1.5, 10], labels=["casi cuadrada (≤1,1)", "moderada (1,1–1,5)", "alargada (>1,5)"])
por_forma = df.groupby("grupo_forma", observed=True).agg(
    imagenes=("acierta_E2", "size"), accuracy_E2=("acierta_E2", "mean")).round(3)
print("Accuracy de E2 según la forma de la foto (cuánto se recorta):"); display(por_forma)

por_brillo.to_csv(f"{SALIDA}/accuracy_por_brillo.csv"); por_forma.to_csv(f"{SALIDA}/accuracy_por_forma.csv")
df.drop(columns=["grupo_brillo", "grupo_forma"]).to_csv(f"{SALIDA}/predicciones_E2_con_confianza.csv", index=False)

Accuracy de E2 según el brillo de la foto:


,imagenes,accuracy_E2,brillo_medio
grupo_brillo,,,
1 más oscuras,469,0.802,0.463
2,470,0.798,0.520
3,473,0.833,0.565
4 más claras,462,0.784,0.638


Accuracy de E2 según la forma de la foto (cuánto se recorta):


,imagenes,accuracy_E2
grupo_forma,,
"casi cuadrada (≤1,1)",270,0.737
"moderada (1,1–1,5)",1185,0.808
"alargada (>1,5)",419,0.835


## Celda 10 · Subir resultados
Ejecútala **después** de completar la celda 7 (y volver a ejecutarla).

In [10]:
!git pull
!git add results/analisis_errores
!git commit -m "Análisis de errores: 12 errores de E2 con Grad-CAM, aciertos, confusiones E0-E4 y pruebas de brillo y recorte"
!git push

Already up to date.
[main 15b4c51] Análisis de errores: 12 errores de E2 con Grad-CAM, aciertos, confusiones E0-E4 y pruebas de brillo y recorte
 12 files changed, 1921 insertions(+)
 create mode 100644 results/analisis_errores/accuracy_por_brillo.csv
 create mode 100644 results/analisis_errores/accuracy_por_forma.csv
 create mode 100644 results/analisis_errores/aciertos_gradcam.csv
 create mode 100644 results/analisis_errores/aciertos_gradcam.png
 create mode 100644 results/analisis_errores/confusiones_frecuentes_E0_E4.csv
 create mode 100644 results/analisis_errores/errores_gradcam_parte1.png
 create mode 100644 results/analisis_errores/errores_gradcam_parte2.png
 create mode 100644 results/analisis_errores/errores_gradcam_parte3.png
 create mode 100644 results/analisis_errores/errores_seleccionados.csv
 create mode 100644 results/analisis_errores/f1_por_clase_E0_E4.csv
 create mode 100644 results/analisis_errores/predicciones_E2_con_confianza.csv
 create mode 100644 results/analisis

## Celda 11 · Subir este notebook
1. **Ctrl + S**. 2. `!ls "/content/drive/MyDrive/Colab Notebooks/"` y reemplaza `NOMBRE_EXACTO`.

In [11]:
%cd /content/yuca_transfer
!cp "/content/drive/MyDrive/Colab Notebooks/NOMBRE_EXACTO.ipynb" notebooks/08_analisis_errores.ipynb
!git add notebooks/08_analisis_errores.ipynb
!git commit -m "Notebook de análisis de errores"
!git push

/content/yuca_transfer
cp: cannot stat '/content/drive/MyDrive/Colab Notebooks/NOMBRE_EXACTO.ipynb': No such file or directory
fatal: pathspec 'notebooks/08_analisis_errores.ipynb' did not match any files
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date
